# 🏥 AI Medical Diagnosis Assistant Agent
### Powered by LangGraph · Groq LLaMA-3.1-8B · Serper API

---

> **Purpose:** Assist doctors with preliminary diagnosis using multi-agent AI workflows  
> **Industry:** Healthcare · Telemedicine  
> **Framework:** LangGraph (Multi-Agent Orchestration)  
> **Model:** `llama-3.1-8b-instant` via Groq  
> **Search:** Serper API for real-time medical web search

---

## 🔄 Workflow Architecture
```
Patient Symptoms → Patient Data Agent
                        ↓
               Medical Knowledge Agent  ← [Serper Web Search + Clinical Guidelines]
                        ↓
                 Diagnosis Agent        ← [NLP + Differential Diagnosis]
                        ↓
        Treatment Recommendation Agent  ← [Medical Databases]
                        ↓
              Final Clinical Report
```

---
⚠️ **DISCLAIMER:** This tool is for educational and research purposes only. Always consult qualified medical professionals for actual medical decisions.

## 📦 Cell 1: Install Required Libraries

In [ ]:
# ============================================================
# CELL 1: Install All Required Libraries
# ============================================================

print("🔧 Installing required libraries...")

!pip install -q langchain langchain-groq langchain-community
!pip install -q langgraph
!pip install -q gradio
!pip install -q requests python-dotenv
!pip install -q pydantic

print("✅ All libraries installed successfully!")

## 📚 Cell 2: Import All Libraries

In [ ]:
# ============================================================
# CELL 2: Import All Necessary Libraries
# ============================================================

# --- Standard Libraries ---
import os
import json
import re
import time
import datetime
from typing import TypedDict, Annotated, List, Dict, Any, Optional, Sequence

# --- LangChain Core ---
from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    AIMessage,
    SystemMessage
)
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langchain_core.output_parsers import StrOutputParser

# --- LangChain Groq ---
from langchain_groq import ChatGroq

# --- LangGraph ---
from langgraph.graph import StateGraph, END, START
from langgraph.prebuilt import ToolNode
from langgraph.graph.message import add_messages

# --- HTTP Requests (Serper) ---
import requests

# --- Pydantic ---
from pydantic import BaseModel, Field

# --- Gradio UI ---
import gradio as gr

print("✅ All imports successful!")
print(f"   📦 LangGraph: Ready")
print(f"   🤖 LangChain-Groq: Ready")
print(f"   🎨 Gradio: Ready")
print(f"   🔧 Pydantic: Ready")

## 🔑 Cell 3: API Configuration

In [ ]:
# ============================================================
# CELL 3: API Keys & Configuration
# ============================================================
# ⚠️  IMPORTANT: Replace with your actual API keys
#     Groq API  → https://console.groq.com/keys
#     Serper API → https://serper.dev/api-key
# ============================================================

# --- 🔑 API Keys (Replace with your actual keys) ---
GROQ_API_KEY    = "your_groq_api_key_here"       # 🔑 Get from: https://console.groq.com
SERPER_API_KEY  = "your_serper_api_key_here"     # 🔑 Get from: https://serper.dev

# --- Set as Environment Variables ---
os.environ["GROQ_API_KEY"]   = GROQ_API_KEY
os.environ["SERPER_API_KEY"] = SERPER_API_KEY

# --- 🤖 Model Configuration ---
MODEL_NAME        = "llama-3.1-8b-instant"   # Groq LLaMA model
MODEL_TEMPERATURE = 0.1                       # Low temp for medical accuracy
MODEL_MAX_TOKENS  = 2048                      # Max output tokens

# --- Validate Keys ---
if GROQ_API_KEY == "your_groq_api_key_here":
    print("⚠️  WARNING: Please set your GROQ_API_KEY before running agents!")
else:
    print(f"✅ Groq API Key configured: {GROQ_API_KEY[:8]}...")

if SERPER_API_KEY == "your_serper_api_key_here":
    print("⚠️  WARNING: Please set your SERPER_API_KEY before running agents!")
else:
    print(f"✅ Serper API Key configured: {SERPER_API_KEY[:8]}...")

print(f"\n🤖 Model: {MODEL_NAME}")
print(f"🌡️  Temperature: {MODEL_TEMPERATURE}")
print(f"📝 Max Tokens: {MODEL_MAX_TOKENS}")

## 🤖 Cell 4: Initialize Groq LLM

In [ ]:
# ============================================================
# CELL 4: Initialize Groq LLM - llama-3.1-8b-instant
# ============================================================

def initialize_llm():
    """Initialize the Groq LLM with llama-3.1-8b-instant model."""
    llm = ChatGroq(
        model=MODEL_NAME,
        temperature=MODEL_TEMPERATURE,
        max_tokens=MODEL_MAX_TOKENS,
        groq_api_key=GROQ_API_KEY,
        verbose=False
    )
    return llm

# Initialize the LLM
llm = initialize_llm()
print(f"✅ Groq LLM Initialized")
print(f"   Model   : {MODEL_NAME}")
print(f"   Provider: Groq (High-Speed Inference)")
print(f"   Speed   : ~750 tokens/sec")

## 🛠️ Cell 5: Define Tools (Serper Web Search + Medical Databases)

In [ ]:
# ============================================================
# CELL 5: Tool Definitions
# ============================================================

# --- Tool 1: Serper Medical Web Search ---
@tool
def medical_web_search(query: str) -> str:
    """
    Search the web for medical information using Serper API.
    Use this to find current medical research, guidelines, and clinical information.
    Args:
        query: Medical search query (e.g., 'hypertension treatment guidelines 2024')
    Returns:
        Formatted search results with medical information
    """
    try:
        url = "https://google.serper.dev/search"
        headers = {
            "X-API-KEY": SERPER_API_KEY,
            "Content-Type": "application/json"
        }
        payload = {
            "q": f"medical {query} clinical guidelines",
            "num": 5,
            "gl": "us",
            "hl": "en"
        }
        response = requests.post(url, headers=headers, json=payload, timeout=10)
        response.raise_for_status()
        data = response.json()

        results = []
        # Extract organic results
        for item in data.get("organic", [])[:4]:
            title   = item.get("title", "No Title")
            snippet = item.get("snippet", "No description")
            link    = item.get("link", "")
            results.append(f"📋 {title}\n   {snippet}\n   Source: {link}")

        # Extract knowledge graph if available
        kg = data.get("knowledgeGraph", {})
        if kg:
            results.insert(0, f"🏥 {kg.get('title', '')}: {kg.get('description', '')}")

        return "\n\n".join(results) if results else "No relevant medical information found."

    except requests.exceptions.ConnectionError:
        return f"[SEARCH SIMULATION] Medical search for '{query}': Based on established medical literature, this condition typically involves standard diagnostic criteria and evidence-based treatment protocols."
    except Exception as e:
        return f"[SEARCH SIMULATION] Medical information for '{query}': Refer to current clinical guidelines and evidence-based medical resources for comprehensive information."


# --- Tool 2: ICD-10 Code Lookup (Simulated Medical Database) ---
@tool
def lookup_icd10_code(condition: str) -> str:
    """
    Look up ICD-10 diagnostic codes for medical conditions.
    Useful for standardized medical coding and billing.
    Args:
        condition: Medical condition name (e.g., 'hypertension', 'diabetes type 2')
    Returns:
        ICD-10 code and description
    """
    icd10_database = {
        "hypertension": "I10 - Essential (primary) hypertension",
        "diabetes type 2": "E11 - Type 2 diabetes mellitus",
        "diabetes type 1": "E10 - Type 1 diabetes mellitus",
        "pneumonia": "J18 - Pneumonia, unspecified organism",
        "asthma": "J45 - Asthma",
        "coronary artery disease": "I25 - Chronic ischaemic heart disease",
        "heart failure": "I50 - Heart failure",
        "atrial fibrillation": "I48 - Atrial fibrillation and flutter",
        "stroke": "I64 - Stroke, not specified",
        "migraine": "G43 - Migraine",
        "depression": "F32 - Depressive episode",
        "anxiety": "F41 - Other anxiety disorders",
        "copd": "J44 - Other chronic obstructive pulmonary disease",
        "kidney disease": "N18 - Chronic kidney disease",
        "hypothyroidism": "E03 - Other hypothyroidism",
        "hyperthyroidism": "E05 - Thyrotoxicosis",
        "anemia": "D64 - Other anemias",
        "sepsis": "A41 - Other sepsis",
        "appendicitis": "K37 - Unspecified appendicitis",
        "gastroenteritis": "K59 - Other functional intestinal disorders",
        "urinary tract infection": "N39 - Other disorders of urinary system",
        "influenza": "J11 - Influenza due to unidentified influenza virus",
        "covid-19": "U07.1 - COVID-19",
        "tuberculosis": "A15 - Respiratory tuberculosis",
        "gout": "M10 - Gout",
        "osteoporosis": "M81 - Osteoporosis without current pathological fracture",
    }
    condition_lower = condition.lower().strip()
    for key, value in icd10_database.items():
        if key in condition_lower or condition_lower in key:
            return f"ICD-10 Code: {value}"
    return f"ICD-10 code for '{condition}' not found in local database. Please consult WHO ICD-10 official classification."


# --- Tool 3: Drug Interaction Checker ---
@tool
def check_drug_interactions(medications: str) -> str:
    """
    Check for potential drug interactions between medications.
    Args:
        medications: Comma-separated list of medications (e.g., 'aspirin, warfarin, ibuprofen')
    Returns:
        Known drug interaction warnings
    """
    known_interactions = {
        ("aspirin", "warfarin"): "⚠️ HIGH RISK: Aspirin + Warfarin → Increased bleeding risk. Monitor INR closely.",
        ("aspirin", "ibuprofen"): "⚠️ MODERATE: Aspirin + Ibuprofen → Reduced antiplatelet effect of aspirin.",
        ("warfarin", "amoxicillin"): "⚠️ MODERATE: Warfarin + Antibiotics → May increase anticoagulant effect.",
        ("metformin", "alcohol"): "⚠️ HIGH RISK: Metformin + Alcohol → Risk of lactic acidosis.",
        ("ssri", "tramadol"): "⚠️ HIGH RISK: SSRI + Tramadol → Risk of serotonin syndrome.",
        ("ace inhibitor", "potassium"): "⚠️ MODERATE: ACE inhibitors + Potassium supplements → Risk of hyperkalemia.",
        ("statin", "amiodarone"): "⚠️ MODERATE: Statins + Amiodarone → Increased risk of myopathy.",
        ("digoxin", "amiodarone"): "⚠️ HIGH RISK: Digoxin + Amiodarone → Increased digoxin toxicity risk.",
    }

    med_list = [m.strip().lower() for m in medications.split(",")]
    warnings = []

    for i, med1 in enumerate(med_list):
        for med2 in med_list[i+1:]:
            for (drug1, drug2), warning in known_interactions.items():
                if (drug1 in med1 or drug1 in med2) and (drug2 in med1 or drug2 in med2):
                    warnings.append(warning)

    if warnings:
        return "Drug Interaction Alerts:\n" + "\n".join(warnings)
    return f"✅ No major known interactions found between: {medications}. Always verify with pharmacist."


# --- Tool 4: Clinical Guidelines Retriever ---
@tool
def get_clinical_guidelines(condition: str) -> str:
    """
    Retrieve evidence-based clinical guidelines for a medical condition.
    Args:
        condition: Medical condition (e.g., 'type 2 diabetes management')
    Returns:
        Key clinical guideline recommendations
    """
    guidelines = {
        "hypertension": """
        📋 JNC 8 / AHA 2023 Hypertension Guidelines:
        • Target BP: <130/80 mmHg (general adults)
        • Target BP: <140/90 mmHg (low-risk adults >65)
        • First-line: Thiazide diuretics, ACE inhibitors, ARBs, CCBs
        • Lifestyle: DASH diet, sodium restriction (<2.3g/day), exercise 150 min/week
        • Black patients: Prefer thiazides + CCBs over ACE inhibitors alone
        """,
        "diabetes": """
        📋 ADA 2024 Diabetes Standards of Care:
        • HbA1c Target: <7% (most adults), individualize based on patient
        • First-line medication: Metformin (if tolerated)
        • Add GLP-1 agonist or SGLT-2 inhibitor if CV disease present
        • BP target: <130/80 mmHg
        • Screening: Lipids, kidney function, retinal exam annually
        • Lifestyle: Medical nutrition therapy + 150 min moderate exercise/week
        """,
        "pneumonia": """
        📋 IDSA/ATS Community-Acquired Pneumonia Guidelines:
        • Assess severity: PSI/PORT score or CURB-65
        • Outpatient (mild): Amoxicillin OR Doxycycline
        • Inpatient (non-ICU): Beta-lactam + Macrolide OR Respiratory fluoroquinolone
        • ICU: Beta-lactam + Azithromycin OR Beta-lactam + Fluoroquinolone
        • Duration: 5 days minimum (can extend based on clinical response)
        • Blood cultures before antibiotics (inpatient)
        """,
        "heart failure": """
        📋 ACC/AHA 2022 Heart Failure Guidelines:
        • HFrEF (EF<40%): ACE-I/ARB/ARNI + Beta-blocker + MRA + SGLT2i
        • Diuretics for fluid management
        • ICD if EF<35% despite optimal therapy
        • Avoid NSAIDs, CCBs (non-dihydropyridine)
        • Sodium restriction: <2g/day
        • Daily weights, fluid monitoring
        """,
        "asthma": """
        📋 GINA 2023 Asthma Guidelines:
        • Stepwise approach based on symptom control
        • Step 1-2: Low-dose ICS-formoterol as needed
        • Step 3: Low-dose ICS-formoterol maintenance + as-needed
        • Step 4-5: Medium/High-dose ICS + LABA, consider biologics
        • Spirometry for diagnosis (FEV1/FVC <0.7 post-bronchodilator)
        • Avoid allergens/triggers
        """,
    }

    condition_lower = condition.lower()
    for key, guideline in guidelines.items():
        if key in condition_lower:
            return guideline.strip()

    return f"Specific guidelines for '{condition}' not in local cache. Recommend searching UpToDate, PubMed, or relevant specialty society guidelines."


# --- Tool 5: Lab Values Reference ---
@tool
def get_lab_reference_ranges(test_name: str) -> str:
    """
    Get normal reference ranges for common laboratory tests.
    Args:
        test_name: Name of the lab test (e.g., 'HbA1c', 'creatinine', 'CBC')
    Returns:
        Normal reference ranges and clinical interpretation
    """
    lab_ranges = {
        "hba1c": "HbA1c: Normal <5.7% | Prediabetes: 5.7-6.4% | Diabetes: ≥6.5%",
        "glucose": "Fasting Glucose: Normal 70-99 mg/dL | Prediabetes: 100-125 | Diabetes: ≥126 mg/dL",
        "creatinine": "Creatinine: Male 0.74-1.35 mg/dL | Female 0.59-1.04 mg/dL",
        "hemoglobin": "Hemoglobin: Male 13.5-17.5 g/dL | Female 12.0-15.5 g/dL",
        "wbc": "WBC: 4,500-11,000 cells/mcL (Elevated >11,000 suggests infection/inflammation)",
        "platelets": "Platelets: 150,000-400,000/mcL",
        "sodium": "Sodium: 136-145 mEq/L",
        "potassium": "Potassium: 3.5-5.0 mEq/L",
        "tsh": "TSH: 0.4-4.0 mIU/L (↑ = Hypothyroidism, ↓ = Hyperthyroidism)",
        "ldl": "LDL Cholesterol: Optimal <100 mg/dL | Near optimal 100-129 | High ≥160 mg/dL",
        "hdl": "HDL Cholesterol: Low risk >60 mg/dL | High risk <40 mg/dL (male), <50 mg/dL (female)",
        "inr": "INR: Normal 0.8-1.2 | Therapeutic (warfarin): 2.0-3.0",
        "alt": "ALT: 7-56 U/L (Elevated suggests liver disease)",
        "ast": "AST: 10-40 U/L",
        "egfr": "eGFR: ≥90 Normal | 60-89 Mild CKD | 30-59 Moderate | 15-29 Severe | <15 Kidney Failure",
        "troponin": "Troponin I: <0.04 ng/mL (Elevated strongly suggests myocardial injury/MI)",
        "bnp": "BNP: <100 pg/mL normal | >400 pg/mL suggests heart failure",
        "crp": "CRP: <1.0 mg/L low risk | 1-3 average risk | >3 mg/L high risk for CVD",
        "ferritin": "Ferritin: Male 24-336 ng/mL | Female 11-307 ng/mL",
        "vitamin d": "Vitamin D (25-OH): Deficiency <20 ng/mL | Insufficient 20-29 | Sufficient 30-100 ng/mL",
        "psa": "PSA: <4 ng/mL generally normal (age-dependent, consult urologist)",
    }

    test_lower = test_name.lower().strip()
    for key, range_info in lab_ranges.items():
        if key in test_lower or test_lower in key:
            return f"🔬 Lab Reference: {range_info}"

    return f"Reference range for '{test_name}' not found. Consult your laboratory's reference values."


# --- Register All Tools ---
ALL_TOOLS = [
    medical_web_search,
    lookup_icd10_code,
    check_drug_interactions,
    get_clinical_guidelines,
    get_lab_reference_ranges
]

print("✅ All Medical Tools Registered:")
for t in ALL_TOOLS:
    print(f"   🔧 {t.name}")

## 🏗️ Cell 6: Define LangGraph State & Agent Prompts

In [ ]:
# ============================================================
# CELL 6: LangGraph State & Agent System Prompts
# ============================================================

# --- LangGraph State Definition ---
class MedicalAgentState(TypedDict):
    """State shared across all medical agents in the graph."""
    # Patient Information
    patient_name       : str
    patient_age        : int
    patient_gender     : str
    patient_weight     : float
    patient_height     : float
    blood_pressure     : str
    heart_rate         : int
    temperature        : float
    oxygen_saturation  : float

    # Medical History
    chief_complaint    : str
    symptoms           : str
    symptom_duration   : str
    medical_history    : str
    current_medications: str
    allergies          : str
    family_history     : str
    lab_results        : str
    urgency_level      : str
    specialty          : str

    # Agent Outputs
    patient_summary    : str
    medical_knowledge  : str
    differential_dx    : str
    treatment_plan     : str
    final_report       : str

    # Control Flow
    messages           : Annotated[List[BaseMessage], add_messages]
    current_agent      : str
    errors             : List[str]


# --- Agent System Prompts ---

PATIENT_DATA_AGENT_PROMPT = """
You are the Patient Data Agent in an AI Medical Diagnosis System.

ROLE: Collect, organize, and summarize patient information for downstream medical analysis.

YOUR TASKS:
1. Analyze all provided patient demographics and vital signs
2. Identify key clinical findings and abnormal values
3. Flag red flags or emergency symptoms requiring immediate attention
4. Create a structured patient summary for the medical team
5. Calculate BMI and assess cardiovascular risk if applicable

OUTPUT FORMAT:
- Patient Profile Summary
- Vital Signs Assessment (normal/abnormal)
- Key Clinical Findings
- Red Flags (if any)
- Risk Stratification

⚠️ DISCLAIMER: This is for educational purposes. Always defer to qualified physicians for actual medical care.
"""

MEDICAL_KNOWLEDGE_AGENT_PROMPT = """
You are the Medical Knowledge Agent in an AI Medical Diagnosis System.

ROLE: Retrieve and synthesize relevant medical knowledge for the patient's presenting symptoms.

YOUR TASKS:
1. Use medical_web_search to find current evidence-based information about the symptoms
2. Use get_clinical_guidelines to retrieve relevant treatment guidelines
3. Use get_lab_reference_ranges to contextualize any lab results
4. Identify pathophysiology underlying the presenting complaints
5. List key diagnostic criteria for likely conditions

SEARCH STRATEGY:
- Search for each major symptom + potential diagnosis
- Look for current clinical guidelines
- Find evidence-based diagnostic criteria

OUTPUT FORMAT:
- Relevant Pathophysiology
- Key Diagnostic Criteria Found
- Clinical Guidelines Summary
- Recommended Investigations

⚠️ DISCLAIMER: For educational support only. Physicians must make final clinical decisions.
"""

DIAGNOSIS_AGENT_PROMPT = """
You are the Diagnosis Agent — a senior physician AI assistant specializing in differential diagnosis.

ROLE: Generate a prioritized differential diagnosis list based on patient data and medical knowledge.

YOUR TASKS:
1. Use lookup_icd10_code for each suspected diagnosis
2. Generate top 3-5 differential diagnoses ranked by probability
3. For each diagnosis, explain:
   - Supporting evidence from patient data
   - Against evidence
   - Required confirmatory tests
4. Identify the MOST LIKELY diagnosis with reasoning
5. Flag diagnoses that MUST NOT be missed (dangerous conditions)

DIFFERENTIAL DIAGNOSIS FORMAT:
1. [Most Likely] Diagnosis Name (ICD-10 Code)
   - Evidence For: ...
   - Evidence Against: ...
   - Confirmatory Tests: ...

⚠️ DISCLAIMER: AI-generated differential diagnosis is for educational support only. Final diagnosis requires physician evaluation.
"""

TREATMENT_AGENT_PROMPT = """
You are the Treatment Recommendation Agent — a clinical pharmacology and therapeutics AI specialist.

ROLE: Generate evidence-based treatment recommendations for the primary diagnosis.

YOUR TASKS:
1. Use get_clinical_guidelines for evidence-based treatment protocols
2. Use check_drug_interactions to verify medication safety if current meds provided
3. Recommend:
   a) Immediate management (if urgent)
   b) Pharmacological therapy (drug name, dose, duration, monitoring)
   c) Non-pharmacological interventions
   d) Lifestyle modifications
   e) Follow-up and monitoring plan
4. Consider patient-specific factors (age, comorbidities, allergies)
5. Suggest specialist referrals if needed

TREATMENT PLAN FORMAT:
🚨 Immediate Actions:
💊 Medications:
🏃 Lifestyle Modifications:
📅 Follow-up Plan:
👨‍⚕️ Referrals:

⚠️ DISCLAIMER: Treatment recommendations are for educational purposes. Prescribing decisions must be made by licensed physicians.
"""

print("✅ LangGraph State & Agent Prompts Defined")
print("   📊 State fields: Patient Info + Medical History + Agent Outputs")
print("   🤖 Agents: PatientData → MedicalKnowledge → Diagnosis → Treatment")

## 🔗 Cell 7: Build LangGraph Multi-Agent Workflow

In [ ]:
# ============================================================
# CELL 7: LangGraph Multi-Agent Workflow Construction
# ============================================================

# --- LLM with Tools ---
llm_with_tools = llm.bind_tools(ALL_TOOLS)

# ─── Agent 1: Patient Data Agent ────────────────────────────
def patient_data_agent(state: MedicalAgentState) -> MedicalAgentState:
    """Processes and summarizes patient demographics, vitals, and history."""
    print("🏥 [Agent 1] Patient Data Agent running...")

    # Calculate BMI
    bmi = "N/A"
    try:
        if state.get("patient_weight") and state.get("patient_height") and state["patient_height"] > 0:
            h_m = state["patient_height"] / 100
            bmi_val = state["patient_weight"] / (h_m ** 2)
            bmi_cat = "Underweight" if bmi_val < 18.5 else "Normal" if bmi_val < 25 else "Overweight" if bmi_val < 30 else "Obese"
            bmi = f"{bmi_val:.1f} ({bmi_cat})"
    except:
        pass

    patient_context = f"""
    PATIENT INFORMATION:
    - Name: {state.get('patient_name', 'Unknown')}
    - Age: {state.get('patient_age', 'N/A')} years | Gender: {state.get('patient_gender', 'N/A')}
    - Weight: {state.get('patient_weight', 'N/A')} kg | Height: {state.get('patient_height', 'N/A')} cm | BMI: {bmi}

    VITAL SIGNS:
    - Blood Pressure: {state.get('blood_pressure', 'N/A')} mmHg
    - Heart Rate: {state.get('heart_rate', 'N/A')} bpm
    - Temperature: {state.get('temperature', 'N/A')} °C
    - O2 Saturation: {state.get('oxygen_saturation', 'N/A')}%

    CHIEF COMPLAINT: {state.get('chief_complaint', 'Not specified')}
    SYMPTOMS: {state.get('symptoms', 'Not specified')}
    DURATION: {state.get('symptom_duration', 'Not specified')}
    URGENCY: {state.get('urgency_level', 'Routine')}
    SPECIALTY: {state.get('specialty', 'General Medicine')}

    MEDICAL HISTORY: {state.get('medical_history', 'None reported')}
    CURRENT MEDICATIONS: {state.get('current_medications', 'None')}
    ALLERGIES: {state.get('allergies', 'NKDA')}
    FAMILY HISTORY: {state.get('family_history', 'None reported')}
    LAB RESULTS: {state.get('lab_results', 'Not available')}
    """

    messages = [
        SystemMessage(content=PATIENT_DATA_AGENT_PROMPT),
        HumanMessage(content=f"Please analyze this patient and create a structured clinical summary:\n{patient_context}")
    ]

    response = llm.invoke(messages)
    patient_summary = response.content

    return {
        **state,
        "patient_summary": patient_summary,
        "current_agent": "patient_data_agent",
        "messages": [AIMessage(content=f"[Patient Data Agent]\n{patient_summary}")]
    }


# ─── Agent 2: Medical Knowledge Agent ───────────────────────
def medical_knowledge_agent(state: MedicalAgentState) -> MedicalAgentState:
    """Retrieves relevant medical knowledge using web search and guidelines."""
    print("📚 [Agent 2] Medical Knowledge Agent running...")

    context = f"""
    Patient Summary: {state.get('patient_summary', '')}

    Chief Complaint: {state.get('chief_complaint', '')}
    Symptoms: {state.get('symptoms', '')}
    Medical History: {state.get('medical_history', '')}
    Lab Results: {state.get('lab_results', 'None')}
    Specialty Focus: {state.get('specialty', 'General Medicine')}
    """

    messages = [
        SystemMessage(content=MEDICAL_KNOWLEDGE_AGENT_PROMPT),
        HumanMessage(content=f"Research medical knowledge for this patient. Use available tools to search for relevant conditions and guidelines:\n{context}")
    ]

    # Agentic loop — let the LLM use tools
    tool_map = {t.name: t for t in ALL_TOOLS}
    current_messages = messages.copy()

    for _ in range(3):  # Max 3 tool-use iterations
        response = llm_with_tools.invoke(current_messages)
        current_messages.append(response)

        if not response.tool_calls:
            break

        for tool_call in response.tool_calls:
            tool_fn = tool_map.get(tool_call["name"])
            if tool_fn:
                try:
                    tool_result = tool_fn.invoke(tool_call["args"])
                except Exception as e:
                    tool_result = f"Tool error: {str(e)}"
                from langchain_core.messages import ToolMessage
                current_messages.append(
                    ToolMessage(content=str(tool_result), tool_call_id=tool_call["id"])
                )

    medical_knowledge = response.content

    return {
        **state,
        "medical_knowledge": medical_knowledge,
        "current_agent": "medical_knowledge_agent",
        "messages": [AIMessage(content=f"[Medical Knowledge Agent]\n{medical_knowledge}")]
    }


# ─── Agent 3: Diagnosis Agent ────────────────────────────────
def diagnosis_agent(state: MedicalAgentState) -> MedicalAgentState:
    """Generates prioritized differential diagnosis list."""
    print("🔬 [Agent 3] Diagnosis Agent running...")

    context = f"""
    Patient Summary: {state.get('patient_summary', '')}
    Medical Knowledge: {state.get('medical_knowledge', '')}

    Symptoms: {state.get('symptoms', '')}
    Duration: {state.get('symptom_duration', '')}
    Medical History: {state.get('medical_history', '')}
    Lab Results: {state.get('lab_results', 'None provided')}
    Vital Signs: BP {state.get('blood_pressure', 'N/A')}, HR {state.get('heart_rate', 'N/A')}, Temp {state.get('temperature', 'N/A')}°C, SpO2 {state.get('oxygen_saturation', 'N/A')}%
    """

    messages = [
        SystemMessage(content=DIAGNOSIS_AGENT_PROMPT),
        HumanMessage(content=f"Generate a differential diagnosis for this patient. Use ICD-10 lookup for each diagnosis:\n{context}")
    ]

    tool_map = {t.name: t for t in ALL_TOOLS}
    current_messages = messages.copy()

    for _ in range(4):
        response = llm_with_tools.invoke(current_messages)
        current_messages.append(response)

        if not response.tool_calls:
            break

        for tool_call in response.tool_calls:
            tool_fn = tool_map.get(tool_call["name"])
            if tool_fn:
                try:
                    tool_result = tool_fn.invoke(tool_call["args"])
                except Exception as e:
                    tool_result = f"Tool error: {str(e)}"
                from langchain_core.messages import ToolMessage
                current_messages.append(
                    ToolMessage(content=str(tool_result), tool_call_id=tool_call["id"])
                )

    differential_dx = response.content

    return {
        **state,
        "differential_dx": differential_dx,
        "current_agent": "diagnosis_agent",
        "messages": [AIMessage(content=f"[Diagnosis Agent]\n{differential_dx}")]
    }


# ─── Agent 4: Treatment Recommendation Agent ─────────────────
def treatment_recommendation_agent(state: MedicalAgentState) -> MedicalAgentState:
    """Generates evidence-based treatment recommendations."""
    print("💊 [Agent 4] Treatment Recommendation Agent running...")

    context = f"""
    Patient: {state.get('patient_name', 'Unknown')}, Age {state.get('patient_age', 'N/A')}, {state.get('patient_gender', 'N/A')}
    Differential Diagnosis: {state.get('differential_dx', '')}
    Current Medications: {state.get('current_medications', 'None')}
    Allergies: {state.get('allergies', 'NKDA')}
    Medical History: {state.get('medical_history', 'None')}
    Urgency Level: {state.get('urgency_level', 'Routine')}
    Specialty: {state.get('specialty', 'General Medicine')}
    """

    messages = [
        SystemMessage(content=TREATMENT_AGENT_PROMPT),
        HumanMessage(content=f"Generate a comprehensive treatment plan. Check drug interactions if medications are listed:\n{context}")
    ]

    tool_map = {t.name: t for t in ALL_TOOLS}
    current_messages = messages.copy()

    for _ in range(4):
        response = llm_with_tools.invoke(current_messages)
        current_messages.append(response)

        if not response.tool_calls:
            break

        for tool_call in response.tool_calls:
            tool_fn = tool_map.get(tool_call["name"])
            if tool_fn:
                try:
                    tool_result = tool_fn.invoke(tool_call["args"])
                except Exception as e:
                    tool_result = f"Tool error: {str(e)}"
                from langchain_core.messages import ToolMessage
                current_messages.append(
                    ToolMessage(content=str(tool_result), tool_call_id=tool_call["id"])
                )

    treatment_plan = response.content

    return {
        **state,
        "treatment_plan": treatment_plan,
        "current_agent": "treatment_recommendation_agent",
        "messages": [AIMessage(content=f"[Treatment Agent]\n{treatment_plan}")]
    }


# ─── Final Report Generator ───────────────────────────────────
def generate_final_report(state: MedicalAgentState) -> MedicalAgentState:
    """Compiles all agent outputs into a final clinical report."""
    print("📋 [Final] Generating Clinical Report...")

    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    final_report = f"""╔══════════════════════════════════════════════════════════════╗
║           AI MEDICAL DIAGNOSIS ASSISTANT - CLINICAL REPORT    ║
║              🏥 For Educational & Support Purposes Only        ║
╚══════════════════════════════════════════════════════════════╝

📅 Report Generated: {timestamp}
🤖 AI Model: {MODEL_NAME} (Groq)
🏥 Specialty: {state.get('specialty', 'General Medicine')}
⚡ Urgency: {state.get('urgency_level', 'Routine')}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SECTION 1: PATIENT SUMMARY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{state.get('patient_summary', 'Not generated')}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SECTION 2: MEDICAL KNOWLEDGE & RESEARCH
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{state.get('medical_knowledge', 'Not generated')}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SECTION 3: DIFFERENTIAL DIAGNOSIS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{state.get('differential_dx', 'Not generated')}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SECTION 4: TREATMENT RECOMMENDATIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{state.get('treatment_plan', 'Not generated')}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
⚠️  IMPORTANT DISCLAIMER
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
This AI-generated report is for EDUCATIONAL and DECISION SUPPORT purposes only.
It does NOT replace clinical judgment of qualified medical professionals.
All diagnoses and treatments must be verified and approved by licensed physicians.
In case of emergency, call 911 or go to the nearest emergency room immediately.
"""

    return {
        **state,
        "final_report": final_report,
        "current_agent": "complete",
        "messages": [AIMessage(content="Clinical report generated successfully.")]
    }


print("✅ All Agent Functions Defined")
print("   Agent 1: patient_data_agent")
print("   Agent 2: medical_knowledge_agent")
print("   Agent 3: diagnosis_agent")
print("   Agent 4: treatment_recommendation_agent")
print("   Final  : generate_final_report")

## 🕸️ Cell 8: Compile LangGraph Workflow

In [ ]:
# ============================================================
# CELL 8: Compile the LangGraph Workflow
# ============================================================

def build_medical_graph():
    """Build and compile the LangGraph medical diagnosis workflow."""

    # Create the state graph
    graph_builder = StateGraph(MedicalAgentState)

    # Add all agent nodes
    graph_builder.add_node("patient_data_agent",             patient_data_agent)
    graph_builder.add_node("medical_knowledge_agent",        medical_knowledge_agent)
    graph_builder.add_node("diagnosis_agent",                diagnosis_agent)
    graph_builder.add_node("treatment_recommendation_agent", treatment_recommendation_agent)
    graph_builder.add_node("generate_final_report",          generate_final_report)

    # Define the sequential workflow edges
    graph_builder.add_edge(START,                            "patient_data_agent")
    graph_builder.add_edge("patient_data_agent",             "medical_knowledge_agent")
    graph_builder.add_edge("medical_knowledge_agent",        "diagnosis_agent")
    graph_builder.add_edge("diagnosis_agent",                "treatment_recommendation_agent")
    graph_builder.add_edge("treatment_recommendation_agent", "generate_final_report")
    graph_builder.add_edge("generate_final_report",          END)

    # Compile the graph
    compiled_graph = graph_builder.compile()
    return compiled_graph


# Build the graph
medical_graph = build_medical_graph()

print("✅ LangGraph Medical Workflow Compiled Successfully!")
print()
print("📊 Workflow Architecture:")
print("   START")
print("     ↓")
print("   [1] Patient Data Agent          → Analyzes vitals & history")
print("     ↓")
print("   [2] Medical Knowledge Agent     → Searches guidelines & evidence")
print("     ↓")
print("   [3] Diagnosis Agent             → Differential diagnosis + ICD-10")
print("     ↓")
print("   [4] Treatment Recommendation    → Evidence-based treatment plan")
print("     ↓")
print("   [5] Final Report Generator      → Clinical report")
print("     ↓")
print("   END")

## 🔧 Cell 9: Helper Functions & State Builder

In [ ]:
# ============================================================
# CELL 9: Helper Functions & State Builder
# ============================================================

def build_initial_state(
    patient_name, patient_age, patient_gender, patient_weight,
    patient_height, blood_pressure, heart_rate, temperature,
    oxygen_saturation, chief_complaint, symptoms, symptom_duration,
    medical_history, current_medications, allergies, family_history,
    lab_results, urgency_level, specialty
) -> MedicalAgentState:
    """Build the initial LangGraph state from user inputs."""
    return MedicalAgentState(
        patient_name       = str(patient_name) if patient_name else "Patient",
        patient_age        = int(patient_age) if patient_age else 0,
        patient_gender     = str(patient_gender) if patient_gender else "Unknown",
        patient_weight     = float(patient_weight) if patient_weight else 0.0,
        patient_height     = float(patient_height) if patient_height else 0.0,
        blood_pressure     = str(blood_pressure) if blood_pressure else "Not recorded",
        heart_rate         = int(heart_rate) if heart_rate else 0,
        temperature        = float(temperature) if temperature else 37.0,
        oxygen_saturation  = float(oxygen_saturation) if oxygen_saturation else 0.0,
        chief_complaint    = str(chief_complaint) if chief_complaint else "Not specified",
        symptoms           = str(symptoms) if symptoms else "Not specified",
        symptom_duration   = str(symptom_duration) if symptom_duration else "Not specified",
        medical_history    = str(medical_history) if medical_history else "None reported",
        current_medications= str(current_medications) if current_medications else "None",
        allergies          = str(allergies) if allergies else "NKDA",
        family_history     = str(family_history) if family_history else "None reported",
        lab_results        = str(lab_results) if lab_results else "Not provided",
        urgency_level      = str(urgency_level) if urgency_level else "Routine",
        specialty          = str(specialty) if specialty else "General Medicine",
        patient_summary    = "",
        medical_knowledge  = "",
        differential_dx    = "",
        treatment_plan     = "",
        final_report       = "",
        messages           = [],
        current_agent      = "start",
        errors             = []
    )


def run_diagnosis_pipeline(
    patient_name, patient_age, patient_gender, patient_weight,
    patient_height, blood_pressure, heart_rate, temperature,
    oxygen_saturation, chief_complaint, symptoms, symptom_duration,
    medical_history, current_medications, allergies, family_history,
    lab_results, urgency_level, specialty
) -> tuple:
    """
    Main pipeline runner — builds state, runs LangGraph, returns results.
    Returns: (patient_summary, medical_knowledge, differential_dx, treatment_plan, final_report)
    """
    try:
        print("\n" + "="*60)
        print("🚀 Starting AI Medical Diagnosis Pipeline")
        print("="*60)

        # Build initial state
        initial_state = build_initial_state(
            patient_name, patient_age, patient_gender, patient_weight,
            patient_height, blood_pressure, heart_rate, temperature,
            oxygen_saturation, chief_complaint, symptoms, symptom_duration,
            medical_history, current_medications, allergies, family_history,
            lab_results, urgency_level, specialty
        )

        start_time = time.time()

        # Run the LangGraph pipeline
        final_state = medical_graph.invoke(initial_state)

        elapsed = time.time() - start_time
        print(f"\n✅ Pipeline completed in {elapsed:.1f} seconds")

        return (
            final_state.get("patient_summary",   "Error: No patient summary generated"),
            final_state.get("medical_knowledge", "Error: No medical knowledge generated"),
            final_state.get("differential_dx",   "Error: No differential diagnosis generated"),
            final_state.get("treatment_plan",    "Error: No treatment plan generated"),
            final_state.get("final_report",      "Error: No final report generated")
        )

    except Exception as e:
        error_msg = f"❌ Pipeline Error: {str(e)}\n\nPlease check your API keys and try again."
        print(error_msg)
        return (error_msg, error_msg, error_msg, error_msg, error_msg)


print("✅ Helper functions ready")
print("   build_initial_state()     → Creates LangGraph state")
print("   run_diagnosis_pipeline()  → Runs full multi-agent pipeline")

## 🎨 Cell 10: Gradio Professional UI

In [ ]:
# ============================================================
# CELL 10: Professional Gradio UI — AI Medical Diagnosis Assistant
# ============================================================

# --- Custom CSS Styling ---
CUSTOM_CSS = """
/* ─── Global Styles ─── */
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&family=Merriweather:wght@300;400;700&display=swap');

body, .gradio-container {
    background: linear-gradient(135deg, #f0f4ff 0%, #e8f4f8 50%, #f0fff4 100%) !important;
    font-family: 'Inter', sans-serif !important;
    min-height: 100vh;
}

/* ─── Header Banner ─── */
.header-banner {
    background: linear-gradient(135deg, #1a3c5e 0%, #0d5e8a 40%, #0a7a6e 100%);
    border-radius: 16px;
    padding: 32px;
    text-align: center;
    margin-bottom: 24px;
    box-shadow: 0 8px 32px rgba(26,60,94,0.25);
    color: white;
}

.header-banner h1 {
    font-family: 'Merriweather', serif;
    font-size: 2.2rem;
    font-weight: 700;
    letter-spacing: -0.5px;
    color: white;
    margin: 0 0 8px 0;
}

.header-banner p {
    color: rgba(255,255,255,0.85);
    font-size: 0.95rem;
    margin: 0;
}

/* ─── Section Headers ─── */
.section-header {
    background: linear-gradient(90deg, #1a3c5e, #0d5e8a);
    color: white !important;
    padding: 10px 18px;
    border-radius: 10px 10px 0 0;
    font-weight: 600;
    font-size: 0.9rem;
    letter-spacing: 0.5px;
    text-transform: uppercase;
}

/* ─── Input Groups ─── */
.input-group {
    background: white;
    border-radius: 12px;
    padding: 20px;
    margin-bottom: 16px;
    box-shadow: 0 2px 12px rgba(0,0,0,0.06);
    border: 1px solid rgba(13,94,138,0.1);
}

/* ─── Gradio Inputs ─── */
.gr-input, .gr-textarea, .gr-dropdown select, input[type='number'] {
    border: 1.5px solid #d1e3ef !important;
    border-radius: 8px !important;
    background: #fafcff !important;
    font-family: 'Inter', sans-serif !important;
    font-size: 0.88rem !important;
    transition: border-color 0.2s ease !important;
}

.gr-input:focus, .gr-textarea:focus {
    border-color: #0d5e8a !important;
    box-shadow: 0 0 0 3px rgba(13,94,138,0.1) !important;
}

/* ─── Buttons ─── */
.btn-diagnose {
    background: linear-gradient(135deg, #1a3c5e, #0d7a6e) !important;
    color: white !important;
    border: none !important;
    border-radius: 12px !important;
    padding: 14px 32px !important;
    font-size: 1.05rem !important;
    font-weight: 600 !important;
    letter-spacing: 0.5px !important;
    cursor: pointer !important;
    box-shadow: 0 4px 15px rgba(26,60,94,0.3) !important;
    transition: all 0.3s ease !important;
    width: 100% !important;
}

.btn-diagnose:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 6px 20px rgba(26,60,94,0.4) !important;
}

.btn-clear {
    background: white !important;
    color: #1a3c5e !important;
    border: 2px solid #1a3c5e !important;
    border-radius: 12px !important;
    font-weight: 600 !important;
    transition: all 0.2s ease !important;
}

.btn-clear:hover {
    background: #f0f4ff !important;
}

/* ─── Tabs ─── */
.tab-nav button {
    font-family: 'Inter', sans-serif !important;
    font-weight: 500 !important;
    color: #556b82 !important;
    border-radius: 8px 8px 0 0 !important;
    transition: all 0.2s ease !important;
}

.tab-nav button.selected {
    background: white !important;
    color: #1a3c5e !important;
    font-weight: 600 !important;
    border-bottom: 3px solid #0d5e8a !important;
}

/* ─── Output Areas ─── */
.output-area {
    background: #fafcff;
    border: 1px solid #d1e3ef;
    border-radius: 10px;
    font-family: 'Inter', sans-serif;
    font-size: 0.88rem;
    line-height: 1.7;
}

/* ─── Status Bar ─── */
.status-bar {
    background: linear-gradient(90deg, #e8f4f8, #f0fff4);
    border: 1px solid #b3d9ec;
    border-radius: 8px;
    padding: 10px 16px;
    font-size: 0.82rem;
    color: #1a3c5e;
    font-weight: 500;
}

/* ─── Warning Banner ─── */
.disclaimer {
    background: linear-gradient(90deg, #fff8e6, #fff3e0);
    border-left: 4px solid #f59e0b;
    border-radius: 0 8px 8px 0;
    padding: 12px 16px;
    font-size: 0.82rem;
    color: #92400e;
    font-weight: 500;
    margin: 12px 0;
}

/* ─── Agent Cards ─── */
.agent-card {
    background: white;
    border-radius: 12px;
    border-left: 4px solid #0d5e8a;
    padding: 16px 20px;
    margin-bottom: 12px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.06);
}

/* ─── Urgency Badge ─── */
.urgency-emergency { color: #dc2626; font-weight: 700; }
.urgency-urgent    { color: #d97706; font-weight: 700; }
.urgency-routine   { color: #059669; font-weight: 600; }

/* ─── Labels ─── */
label {
    font-weight: 500 !important;
    font-size: 0.82rem !important;
    color: #374151 !important;
    text-transform: uppercase !important;
    letter-spacing: 0.3px !important;
}
"""


# --- Sample Patient Cases ---
SAMPLE_CASES = {
    "Case 1: Chest Pain (Cardiology)": {
        "name": "John Mitchell", "age": 58, "gender": "Male",
        "weight": 92.0, "height": 178.0,
        "bp": "155/95", "hr": 88, "temp": 37.2, "spo2": 97.0,
        "complaint": "Chest pain radiating to left arm",
        "symptoms": "Crushing chest pain, left arm radiation, diaphoresis, shortness of breath, nausea. Worsens with exertion.",
        "duration": "Started 2 hours ago, progressively worsening",
        "history": "Hypertension (10 years), Hyperlipidemia, Smoker (30 pack-years)",
        "meds": "Amlodipine 10mg, Atorvastatin 40mg, Aspirin 81mg",
        "allergies": "Penicillin (rash)",
        "family": "Father died of MI at age 62, mother has T2DM",
        "labs": "Troponin I: 0.8 ng/mL (elevated), ECG: ST elevation in V1-V4",
        "urgency": "🚨 Emergency",
        "specialty": "Cardiology"
    },
    "Case 2: Diabetes Management (Endocrinology)": {
        "name": "Maria Santos", "age": 52, "gender": "Female",
        "weight": 78.0, "height": 162.0,
        "bp": "138/82", "hr": 76, "temp": 37.0, "spo2": 98.0,
        "complaint": "Poorly controlled diabetes with fatigue",
        "symptoms": "Excessive thirst, frequent urination, fatigue, blurred vision, slow-healing foot wound",
        "duration": "Symptoms progressively worsening over 3 months",
        "history": "Type 2 Diabetes (8 years), Obesity, HTN",
        "meds": "Metformin 1000mg BID, Lisinopril 10mg",
        "allergies": "NKDA",
        "family": "Mother and 2 siblings with T2DM",
        "labs": "HbA1c: 9.2%, FBG: 245 mg/dL, eGFR: 72, Urine albumin: 45 mg/g",
        "urgency": "⚠️ Urgent",
        "specialty": "Endocrinology"
    },
    "Case 3: Respiratory (Pulmonology)": {
        "name": "David Chen", "age": 34, "gender": "Male",
        "weight": 70.0, "height": 172.0,
        "bp": "118/76", "hr": 98, "temp": 38.5, "spo2": 94.0,
        "complaint": "Fever, cough, and shortness of breath",
        "symptoms": "Productive cough with yellow sputum, fever, chills, pleuritic chest pain, dyspnea on exertion",
        "duration": "5 days, worsening last 48 hours",
        "history": "Asthma (mild, controlled), Non-smoker",
        "meds": "Albuterol MDI PRN, Fluticasone 100mcg BID",
        "allergies": "Sulfa drugs (anaphylaxis)",
        "family": "No significant family history",
        "labs": "WBC: 14,500, CRP: 68 mg/L, CXR: Right lower lobe consolidation",
        "urgency": "⚠️ Urgent",
        "specialty": "Pulmonology"
    },
    "Case 4: Headache (Neurology)": {
        "name": "Sarah Johnson", "age": 29, "gender": "Female",
        "weight": 62.0, "height": 165.0,
        "bp": "122/78", "hr": 72, "temp": 37.1, "spo2": 99.0,
        "complaint": "Severe recurrent headaches",
        "symptoms": "Unilateral throbbing headache, photophobia, phonophobia, nausea, visual aura (zigzag lines) before onset",
        "duration": "Recurring for 2 years, current episode 6 hours",
        "history": "No significant past medical history, OCP use",
        "meds": "Oral contraceptive pill, occasional Ibuprofen",
        "allergies": "NKDA",
        "family": "Mother has migraines",
        "labs": "Normal CBC, BMP. MRI brain (6 months ago): Normal",
        "urgency": "📅 Routine",
        "specialty": "Neurology"
    }
}


# --- Load Sample Case Function ---
def load_sample_case(case_name):
    """Load a predefined sample patient case."""
    if case_name == "-- Select a Sample Case --" or not case_name:
        return [gr.update()] * 20

    case = SAMPLE_CASES.get(case_name, {})
    if not case:
        return [gr.update()] * 20

    urgency_map = {"🚨 Emergency": "🚨 Emergency", "⚠️ Urgent": "⚠️ Urgent", "📅 Routine": "📅 Routine"}

    return [
        case.get("name", ""),
        case.get("age", 30),
        case.get("gender", "Male"),
        case.get("weight", 70.0),
        case.get("height", 170.0),
        case.get("bp", "120/80"),
        case.get("hr", 72),
        case.get("temp", 37.0),
        case.get("spo2", 98.0),
        case.get("complaint", ""),
        case.get("symptoms", ""),
        case.get("duration", ""),
        case.get("history", ""),
        case.get("meds", ""),
        case.get("allergies", "NKDA"),
        case.get("family", ""),
        case.get("labs", ""),
        urgency_map.get(case.get("urgency", "📅 Routine"), "📅 Routine"),
        case.get("specialty", "General Medicine")
    ]


# ─── Build Gradio Interface ───────────────────────────────────
with gr.Blocks(
    css=CUSTOM_CSS,
    title="AI Medical Diagnosis Assistant",
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="teal",
        neutral_hue="slate",
        font=[gr.themes.GoogleFont("Inter"), "sans-serif"]
    )
) as demo:

    # ─── Header ────────────────────────────────────────────────
    gr.HTML("""
    <div class="header-banner">
        <h1>🏥 AI Medical Diagnosis Assistant</h1>
        <p>
            Multi-Agent System powered by <strong>LangGraph</strong> · 
            <strong>LLaMA 3.1-8B</strong> via Groq · 
            <strong>Serper</strong> Medical Search
        </p>
        <p style="margin-top:8px; font-size:0.78rem; opacity:0.75;">
            Healthcare AI · Telemedicine · Clinical Decision Support
        </p>
    </div>
    """)

    # ─── Disclaimer ────────────────────────────────────────────
    gr.HTML("""
    <div class="disclaimer">
        ⚠️ <strong>Medical Disclaimer:</strong> This AI tool is for <strong>educational and decision-support purposes only</strong>.
        It does NOT replace the clinical judgment of qualified medical professionals.
        All diagnoses and treatment decisions must be made by licensed physicians.
        <strong>In emergencies, call 911 immediately.</strong>
    </div>
    """)

    # ─── Sample Cases & Quick Load ──────────────────────────────
    with gr.Row():
        sample_case_dd = gr.Dropdown(
            choices=["-- Select a Sample Case --"] + list(SAMPLE_CASES.keys()),
            value="-- Select a Sample Case --",
            label="⚡ Quick Load Sample Patient Case",
            interactive=True,
            scale=3
        )
        gr.HTML('<div style="padding:8px; color:#556b82; font-size:0.82rem;">← Select a pre-built case to auto-fill all fields</div>')

    gr.HTML('<hr style="border:none; border-top:1px solid #d1e3ef; margin:16px 0;">')

    # ─── Main Input Area ────────────────────────────────────────
    with gr.Row(equal_height=False):

        # === LEFT COLUMN: Patient Info + Vitals ===
        with gr.Column(scale=1):

            # Patient Demographics
            gr.HTML('<div class="section-header">👤 Patient Demographics</div>')
            with gr.Group():
                patient_name = gr.Textbox(
                    label="Patient Full Name",
                    placeholder="e.g., John Smith",
                    value=""
                )
                with gr.Row():
                    patient_age = gr.Number(
                        label="Age (years)",
                        value=35,
                        minimum=0, maximum=120, precision=0
                    )
                    patient_gender = gr.Dropdown(
                        label="Biological Sex",
                        choices=["Male", "Female", "Other"],
                        value="Male"
                    )
                with gr.Row():
                    patient_weight = gr.Number(
                        label="Weight (kg)",
                        value=70.0,
                        minimum=1, maximum=500, precision=1
                    )
                    patient_height = gr.Number(
                        label="Height (cm)",
                        value=170.0,
                        minimum=50, maximum=250, precision=1
                    )

            gr.HTML('<br>')

            # Vital Signs
            gr.HTML('<div class="section-header">💓 Vital Signs</div>')
            with gr.Group():
                blood_pressure = gr.Textbox(
                    label="Blood Pressure (mmHg)",
                    placeholder="e.g., 120/80",
                    value="120/80"
                )
                with gr.Row():
                    heart_rate = gr.Number(
                        label="Heart Rate (bpm)",
                        value=72,
                        minimum=20, maximum=300, precision=0
                    )
                    temperature = gr.Number(
                        label="Temperature (°C)",
                        value=37.0,
                        minimum=30, maximum=45, precision=1
                    )
                oxygen_saturation = gr.Slider(
                    label="O₂ Saturation (%)",
                    minimum=70, maximum=100,
                    value=98,
                    step=0.5
                )

            gr.HTML('<br>')

            # Triage & Specialty
            gr.HTML('<div class="section-header">🎯 Triage & Specialty</div>')
            with gr.Group():
                urgency_level = gr.Radio(
                    label="Clinical Urgency Level",
                    choices=["🚨 Emergency", "⚠️ Urgent", "📅 Routine"],
                    value="📅 Routine"
                )
                specialty = gr.Dropdown(
                    label="Medical Specialty",
                    choices=[
                        "General Medicine", "Cardiology", "Pulmonology",
                        "Endocrinology", "Neurology", "Gastroenterology",
                        "Nephrology", "Hematology", "Oncology",
                        "Rheumatology", "Infectious Disease", "Emergency Medicine",
                        "Psychiatry", "Dermatology", "Orthopedics"
                    ],
                    value="General Medicine"
                )

        # === RIGHT COLUMN: Clinical Information ===
        with gr.Column(scale=1):

            # Chief Complaint & Symptoms
            gr.HTML('<div class="section-header">🩺 Chief Complaint & Symptoms</div>')
            with gr.Group():
                chief_complaint = gr.Textbox(
                    label="Chief Complaint",
                    placeholder="Primary reason for visit (e.g., Chest pain and shortness of breath)",
                    lines=2
                )
                symptoms = gr.Textbox(
                    label="Detailed Symptoms",
                    placeholder="Describe all symptoms in detail: location, character, severity (1-10), radiation, associated features...",
                    lines=4
                )
                symptom_duration = gr.Dropdown(
                    label="Duration of Symptoms",
                    choices=[
                        "< 1 hour", "1-6 hours", "6-24 hours",
                        "1-3 days", "3-7 days", "1-4 weeks",
                        "1-3 months", "3-12 months", "> 1 year",
                        "Chronic/Recurrent"
                    ],
                    value="1-3 days",
                    allow_custom_value=True
                )

            gr.HTML('<br>')

            # Medical History
            gr.HTML('<div class="section-header">📂 Medical History</div>')
            with gr.Group():
                medical_history = gr.Textbox(
                    label="Past Medical History",
                    placeholder="Previous diagnoses, surgeries, hospitalizations (e.g., HTN 5 years, T2DM, MI 2020)",
                    lines=3
                )
                current_medications = gr.Textbox(
                    label="Current Medications",
                    placeholder="List all medications with doses (e.g., Metformin 500mg BID, Aspirin 81mg daily)",
                    lines=2
                )
                with gr.Row():
                    allergies = gr.Textbox(
                        label="Allergies & Reactions",
                        placeholder="Drug/food allergies (e.g., Penicillin - rash) or NKDA",
                        value="NKDA"
                    )
                    family_history = gr.Textbox(
                        label="Family History",
                        placeholder="Relevant family history (e.g., Father - MI age 55, Mother - DM)"
                    )

            gr.HTML('<br>')

            # Lab & Investigations
            gr.HTML('<div class="section-header">🔬 Lab Results & Investigations</div>')
            with gr.Group():
                lab_results = gr.Textbox(
                    label="Lab Results / Investigations",
                    placeholder="""CBC: WBC 12,000, Hgb 11.2\nBMP: Na 138, K 4.1, Creatinine 1.1\nHbA1c: 7.8%\nCXR: Clear\nECG: Sinus rhythm""",
                    lines=4
                )

    gr.HTML('<br>')

    # ─── Action Buttons ─────────────────────────────────────────
    with gr.Row():
        with gr.Column(scale=3):
            diagnose_btn = gr.Button(
                "🚀 Run AI Medical Diagnosis (4 Agents)",
                variant="primary",
                elem_classes=["btn-diagnose"],
                size="lg"
            )
        with gr.Column(scale=1):
            clear_btn = gr.Button(
                "🔄 Clear All",
                variant="secondary",
                elem_classes=["btn-clear"]
            )

    # ─── Agent Progress Indicators ──────────────────────────────
    gr.HTML("""
    <div style="background:white; border-radius:12px; padding:16px; margin:16px 0;
                box-shadow:0 2px 12px rgba(0,0,0,0.06); border:1px solid #d1e3ef;">
        <div style="font-weight:600; color:#1a3c5e; margin-bottom:12px; font-size:0.88rem;">
            🤖 MULTI-AGENT PIPELINE — LangGraph Orchestration
        </div>
        <div style="display:flex; gap:8px; flex-wrap:wrap;">
            <div style="flex:1; min-width:120px; background:#e8f4f8; border-radius:8px; padding:10px;
                        text-align:center; border-left:3px solid #0d5e8a;">
                <div style="font-size:1.4rem;">👤</div>
                <div style="font-weight:600; font-size:0.78rem; color:#1a3c5e;">Agent 1</div>
                <div style="font-size:0.72rem; color:#556b82;">Patient Data</div>
            </div>
            <div style="display:flex; align-items:center; color:#0d5e8a; font-weight:700;">→</div>
            <div style="flex:1; min-width:120px; background:#e8f8f0; border-radius:8px; padding:10px;
                        text-align:center; border-left:3px solid #0a7a6e;">
                <div style="font-size:1.4rem;">📚</div>
                <div style="font-weight:600; font-size:0.78rem; color:#0a4a3e;">Agent 2</div>
                <div style="font-size:0.72rem; color:#556b82;">Medical Knowledge</div>
            </div>
            <div style="display:flex; align-items:center; color:#0d5e8a; font-weight:700;">→</div>
            <div style="flex:1; min-width:120px; background:#f0e8f8; border-radius:8px; padding:10px;
                        text-align:center; border-left:3px solid #7c3aed;">
                <div style="font-size:1.4rem;">🔬</div>
                <div style="font-weight:600; font-size:0.78rem; color:#4c1d95;">Agent 3</div>
                <div style="font-size:0.72rem; color:#556b82;">Diagnosis</div>
            </div>
            <div style="display:flex; align-items:center; color:#0d5e8a; font-weight:700;">→</div>
            <div style="flex:1; min-width:120px; background:#fff8e6; border-radius:8px; padding:10px;
                        text-align:center; border-left:3px solid #d97706;">
                <div style="font-size:1.4rem;">💊</div>
                <div style="font-weight:600; font-size:0.78rem; color:#92400e;">Agent 4</div>
                <div style="font-size:0.72rem; color:#556b82;">Treatment</div>
            </div>
        </div>
    </div>
    """)

    # ─── Output Tabs ─────────────────────────────────────────────
    gr.HTML('<div class="section-header">📊 AI Diagnostic Results</div>')
    with gr.Tabs():

        with gr.Tab("👤 Patient Summary", id="tab1"):
            gr.HTML('<p style="color:#556b82; font-size:0.82rem; margin:8px 0 12px;">📋 Structured patient assessment from Agent 1 — includes BMI, vital sign analysis, and clinical flags</p>')
            output_patient = gr.Textbox(
                label="Patient Data Agent Output",
                lines=18,
                interactive=False,
                placeholder="Patient summary will appear here after running the diagnosis pipeline..."
            )

        with gr.Tab("📚 Medical Knowledge", id="tab2"):
            gr.HTML('<p style="color:#556b82; font-size:0.82rem; margin:8px 0 12px;">🌐 Evidence-based medical research from Agent 2 — uses Serper web search + clinical guidelines database</p>')
            output_knowledge = gr.Textbox(
                label="Medical Knowledge Agent Output",
                lines=18,
                interactive=False,
                placeholder="Medical knowledge and research will appear here..."
            )

        with gr.Tab("🔬 Differential Diagnosis", id="tab3"):
            gr.HTML('<p style="color:#556b82; font-size:0.82rem; margin:8px 0 12px;">🩺 Prioritized differential diagnosis with ICD-10 codes from Agent 3 — evidence-based reasoning for each possibility</p>')
            output_diagnosis = gr.Textbox(
                label="Diagnosis Agent Output",
                lines=18,
                interactive=False,
                placeholder="Differential diagnosis list will appear here..."
            )

        with gr.Tab("💊 Treatment Plan", id="tab4"):
            gr.HTML('<p style="color:#556b82; font-size:0.82rem; margin:8px 0 12px;">🏥 Evidence-based treatment recommendations from Agent 4 — medications, lifestyle modifications, and follow-up planning</p>')
            output_treatment = gr.Textbox(
                label="Treatment Recommendation Agent Output",
                lines=18,
                interactive=False,
                placeholder="Treatment recommendations will appear here..."
            )

        with gr.Tab("📋 Full Clinical Report", id="tab5"):
            gr.HTML('<p style="color:#556b82; font-size:0.82rem; margin:8px 0 12px;">📄 Complete AI-generated clinical report — all agent outputs compiled into a structured medical document</p>')
            output_report = gr.Textbox(
                label="Complete Clinical Report",
                lines=25,
                interactive=False,
                placeholder="Full clinical report will appear here..."
            )

    # ─── Footer ─────────────────────────────────────────────────
    gr.HTML("""
    <div style="text-align:center; margin-top:24px; padding:16px;
                background:white; border-radius:12px; border:1px solid #d1e3ef;
                box-shadow:0 2px 8px rgba(0,0,0,0.04);">
        <div style="display:flex; justify-content:center; gap:24px; flex-wrap:wrap; margin-bottom:12px;">
            <span style="color:#556b82; font-size:0.8rem;">🤖 <strong>Model:</strong> llama-3.1-8b-instant</span>
            <span style="color:#556b82; font-size:0.8rem;">⚡ <strong>Inference:</strong> Groq (Ultra-fast)</span>
            <span style="color:#556b82; font-size:0.8rem;">🕸️ <strong>Framework:</strong> LangGraph</span>
            <span style="color:#556b82; font-size:0.8rem;">🔍 <strong>Search:</strong> Serper API</span>
            <span style="color:#556b82; font-size:0.8rem;">🤖 <strong>Agents:</strong> 4 Specialized AI Agents</span>
        </div>
        <p style="color:#9ca3af; font-size:0.75rem; margin:0;">
            Built with LangGraph Multi-Agent Orchestration · For Healthcare AI Research & Education
        </p>
    </div>
    """)

    # ─── All Input Components List (for event binding) ──────────
    all_inputs = [
        patient_name, patient_age, patient_gender, patient_weight,
        patient_height, blood_pressure, heart_rate, temperature,
        oxygen_saturation, chief_complaint, symptoms, symptom_duration,
        medical_history, current_medications, allergies, family_history,
        lab_results, urgency_level, specialty
    ]

    all_outputs = [
        output_patient, output_knowledge, output_diagnosis,
        output_treatment, output_report
    ]

    # ─── Event Handlers ──────────────────────────────────────────

    # Diagnose button click
    diagnose_btn.click(
        fn=run_diagnosis_pipeline,
        inputs=all_inputs,
        outputs=all_outputs,
        show_progress=True
    )

    # Load sample case
    sample_case_dd.change(
        fn=load_sample_case,
        inputs=[sample_case_dd],
        outputs=all_inputs[:-1]  # all except specialty (19 fields → 18 sample fields)
    )

    # Clear button
    def clear_all():
        return (
            "", 35, "Male", 70.0, 170.0, "120/80", 72, 37.0, 98.0,
            "", "", "1-3 days", "", "", "NKDA", "", "",
            "📅 Routine", "General Medicine",
            "", "", "", "", ""
        )

    clear_btn.click(
        fn=clear_all,
        inputs=[],
        outputs=all_inputs + all_outputs
    )

print("✅ Gradio UI Built Successfully!")
print("   🎨 Professional UI with light medical theme")
print("   📋 4 sample patient cases available")
print("   📊 5 output tabs (Patient Summary → Report)")
print("   🎛️  Interactive dropdowns, sliders, radio buttons")

## 🚀 Cell 11: Launch the Application

In [ ]:
# ============================================================
# CELL 11: Launch the Gradio Application
# ============================================================
# ⚠️  IMPORTANT: Make sure you've set your API keys in Cell 3!
#     GROQ_API_KEY   = "gsk_..."
#     SERPER_API_KEY = "..."
# ============================================================

print("🚀 Launching AI Medical Diagnosis Assistant...")
print("="*50)
print(f"  Model     : {MODEL_NAME}")
print(f"  Framework : LangGraph (Multi-Agent)")
print(f"  Agents    : 4 (Patient → Knowledge → Diagnosis → Treatment)")
print(f"  Tools     : 5 (Search, ICD-10, Drug Interactions, Guidelines, Labs)")
print("="*50)
print()
print("⚠️  Verify your API keys are set correctly before using!")
print("📌 Quick tip: Load a sample case using the dropdown at the top!")
print()

demo.launch(
    debug=False,
    share=True,           # Creates public URL for sharing
    show_error=True,
    server_name="0.0.0.0",
    server_port=7860
)

---

## 📖 Architecture Reference

### 🕸️ LangGraph Workflow
```
START
  │
  ▼
┌─────────────────────────────────────────────────┐
│  Agent 1: Patient Data Agent                    │
│  • Parse demographics & vitals                  │
│  • Calculate BMI                                │
│  • Flag abnormal findings & red flags           │
│  • Risk stratification                          │
└──────────────────┬──────────────────────────────┘
                   │
                   ▼
┌─────────────────────────────────────────────────┐
│  Agent 2: Medical Knowledge Agent               │
│  Tools: medical_web_search (Serper API)         │
│         get_clinical_guidelines                 │
│         get_lab_reference_ranges                │
│  • Evidence-based research                      │
│  • Current treatment guidelines                 │
└──────────────────┬──────────────────────────────┘
                   │
                   ▼
┌─────────────────────────────────────────────────┐
│  Agent 3: Diagnosis Agent                       │
│  Tools: lookup_icd10_code                       │
│  • Top 3-5 differential diagnoses               │
│  • Evidence for/against each                    │
│  • ICD-10 codes                                 │
│  • "Must not miss" diagnoses flagged            │
└──────────────────┬──────────────────────────────┘
                   │
                   ▼
┌─────────────────────────────────────────────────┐
│  Agent 4: Treatment Recommendation Agent        │
│  Tools: get_clinical_guidelines                 │
│         check_drug_interactions                 │
│  • Evidence-based pharmacotherapy               │
│  • Drug interaction screening                   │
│  • Lifestyle modifications                      │
│  • Follow-up & monitoring plan                  │
└──────────────────┬──────────────────────────────┘
                   │
                   ▼
┌─────────────────────────────────────────────────┐
│  Final Report Generator                         │
│  • Compiles all agent outputs                   │
│  • Structured clinical document                 │
└─────────────────────────────────────────────────┘
  │
  ▼
 END
```

### 🛠️ Tools Summary
| Tool | Purpose |
|------|---------|
| `medical_web_search` | Serper API → Real-time medical research |
| `lookup_icd10_code` | ICD-10 diagnostic code database |
| `check_drug_interactions` | Drug-drug interaction screening |
| `get_clinical_guidelines` | Evidence-based clinical protocols |
| `get_lab_reference_ranges` | Normal lab value reference |

### ⚠️ Disclaimer
> This system is designed for **educational purposes** and **clinical decision support** only.  
> It does **not** replace the judgment of qualified medical professionals.  
> Always consult licensed physicians for actual medical decisions.